# HMSAR-Net: Hierarchical Multi-Scale Adaptive Residual Network
## Subject-Independent sEMG Gesture Recognition on Ninapro DB1

**Architecture Features:**
- Multi-scale temporal feature extraction
- Channel synergy modeling
- Attention-based feature fusion
- Hierarchical residual blocks
- Deep supervision
- Adaptive global pooling

**Dataset:** Ninapro DB1 Exercise 2 (17 gestures, 27 subjects)

**Evaluation:** Subject-independent (Train: s1-s20, Val: s21-s24, Test: s25-s27)

---
## Step 1: Import Libraries

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import scipy.signal as signal
from scipy.signal import butter, filtfilt, iirnotch
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import backend as K
import time
import gc

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

---
## Step 2: Configure Paths

In [ ]:
# CHANGE THIS TO YOUR NINAPRO DATASET PATH
DATA_PATH = "/Users/prajitbaskaran/Documents/EMG/Ninapro Dataset"

# Output directories
PREPROCESSED_DATA_DIR = './preprocessed_data'
CHECKPOINT_DIR = './model_checkpoints'
RESULTS_DIR = './results'

# Create directories
os.makedirs(PREPROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("✅ Paths configured")
print(f"  Data path: {DATA_PATH}")
print(f"  Preprocessed data: {os.path.abspath(PREPROCESSED_DATA_DIR)}")
print(f"  Checkpoints: {os.path.abspath(CHECKPOINT_DIR)}")
print(f"  Results: {os.path.abspath(RESULTS_DIR)}")

---
## Step 3: Data Loading Functions

In [ ]:
def load_ninapro_subject(subject_num, exercise_num=2):
    """
    Load Ninapro DB1 data for a specific subject and exercise
    """
    subject_folder = f"s{subject_num}"
    file_name = f"S{subject_num}_A1_E{exercise_num}.mat"
    file_path = os.path.join(DATA_PATH, subject_folder, file_name)
    
    print(f"Loading: {file_path}")
    
    mat_data = sio.loadmat(file_path)
    
    emg = mat_data['emg']
    restimulus = mat_data['restimulus'].flatten()
    rerepetition = mat_data['rerepetition'].flatten()
    
    return emg, restimulus, rerepetition


def load_all_subjects_exercise2(subject_list):
    """
    Load Exercise 2 data for multiple subjects
    """
    all_emg = []
    all_labels = []
    all_subjects = []
    
    for subject_num in subject_list:
        try:
            emg, restimulus, rerepetition = load_ninapro_subject(subject_num, exercise_num=2)
            
            # Remove rest class (label 0)
            mask = restimulus > 0
            emg_active = emg[mask]
            labels_active = restimulus[mask] - 1  # Shift to 0-16 range
            
            all_emg.append(emg_active)
            all_labels.append(labels_active)
            all_subjects.append(np.full(len(labels_active), subject_num))
            
            print(f"✓ Loaded subject {subject_num:2d}: {len(emg_active):7d} samples, "
                  f"{len(np.unique(labels_active))} gestures")
            
        except Exception as e:
            print(f"✗ Error loading subject {subject_num}: {str(e)}")
    
    all_emg = np.vstack(all_emg)
    all_labels = np.concatenate(all_labels)
    all_subjects = np.concatenate(all_subjects)
    
    print("\n" + "="*60)
    print(f"TOTAL DATA LOADED:")
    print(f"  - EMG shape: {all_emg.shape}")
    print(f"  - Labels shape: {all_labels.shape}")
    print(f"  - Subjects: {len(subject_list)}")
    print(f"  - Unique gestures: {len(np.unique(all_labels))}")
    print("="*60)
    
    return all_emg, all_labels, all_subjects

print("✅ Data loading functions defined")

---
## Step 4: Load Dataset (Subject-Independent Split)

In [ ]:
# Define subject splits for subject-independent evaluation
TRAIN_SUBJECTS = list(range(1, 21))  # s1-s20
VAL_SUBJECTS = list(range(21, 25))   # s21-s24
TEST_SUBJECTS = list(range(25, 28))  # s25-s27

print("\n📊 SUBJECT-INDEPENDENT SPLIT:")
print(f"Training subjects: {TRAIN_SUBJECTS}")
print(f"Validation subjects: {VAL_SUBJECTS}")
print(f"Test subjects: {TEST_SUBJECTS}")
print()

# Load training data
print("\n🔄 Loading TRAINING data...")
train_emg, train_labels, train_subjects = load_all_subjects_exercise2(TRAIN_SUBJECTS)

print("\n🔄 Loading VALIDATION data...")
val_emg, val_labels, val_subjects = load_all_subjects_exercise2(VAL_SUBJECTS)

print("\n🔄 Loading TEST data...")
test_emg, test_labels, test_subjects = load_all_subjects_exercise2(TEST_SUBJECTS)

---
## Step 5: Preprocessing Functions

In [ ]:
def butter_bandpass_filter(data, lowcut=5, highcut=45, fs=100, order=4):
    """
    Apply Butterworth bandpass filter (5-45 Hz)
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    
    if high >= 1.0:
        raise ValueError(f"High cutoff {highcut}Hz exceeds Nyquist {nyq}Hz")
    
    b, a = butter(order, [low, high], btype='band')
    filtered_data = filtfilt(b, a, data, axis=0)
    return filtered_data


def notch_filter(data, freq=50, fs=100, quality=30):
    """
    Apply notch filter to remove power line interference
    """
    freq = 49 if freq >= fs/2 else freq
    b, a = iirnotch(freq, quality, fs)
    filtered_data = filtfilt(b, a, data, axis=0)
    return filtered_data


def preprocess_emg(emg_data, fs=100):
    """
    Complete preprocessing pipeline
    """
    print("Applying bandpass filter (5-45 Hz)...")
    filtered = butter_bandpass_filter(emg_data, lowcut=5, highcut=45, fs=fs)
    
    print("Applying notch filter (49 Hz)...")
    filtered = notch_filter(filtered, freq=49, fs=fs)
    
    return filtered


def transform_to_rng(emg_data, alpha=2**13, beta=0.75):
    """
    Transform sEMG to Raw Numerical Grayscale (RNG)
    """
    transformed = (np.abs(emg_data) * alpha) ** beta
    transformed = np.clip(transformed, 1, 255)
    transformed[transformed < 1] = 0
    normalized = transformed / 255.0
    return normalized


def create_sliding_windows(emg_data, labels, window_size=256, step_size=50, fs=100):
    """
    Create sliding windows from continuous EMG data
    """
    num_samples, num_channels = emg_data.shape
    windows = []
    window_labels = []
    
    for start_idx in range(0, num_samples - window_size + 1, step_size):
        end_idx = start_idx + window_size
        window = emg_data[start_idx:end_idx, :]
        
        window_label_segment = labels[start_idx:end_idx]
        unique_labels, counts = np.unique(window_label_segment, return_counts=True)
        majority_label = unique_labels[np.argmax(counts)]
        
        if np.max(counts) / window_size >= 0.7:
            windows.append(window)
            window_labels.append(majority_label)
    
    windows = np.array(windows)
    window_labels = np.array(window_labels)
    
    return windows, window_labels


def preprocess_dataset(emg_data, labels, set_name="Dataset", 
                       window_size=256, step_size=50, 
                       alpha=2**13, beta=0.75, fs=100):
    """
    Complete preprocessing pipeline for a dataset
    """
    print(f"\n{'='*70}")
    print(f"PREPROCESSING {set_name}")
    print(f"{'='*70}")
    
    print(f"Input shape: {emg_data.shape}")
    print(f"Window size: {window_size} samples ({window_size/fs:.2f}s)")
    print(f"Step size: {step_size} samples ({step_size/fs:.2f}s)")
    
    # Step 1: Filter
    print("\n[1/3] Filtering...")
    filtered_data = preprocess_emg(emg_data, fs=fs)
    
    # Step 2: Windowing
    print("\n[2/3] Creating sliding windows...")
    windows, window_labels = create_sliding_windows(
        filtered_data, labels, 
        window_size=window_size, 
        step_size=step_size, 
        fs=fs
    )
    print(f"  ✓ Created {len(windows)} windows")
    print(f"  ✓ Window shape: {windows.shape}")
    
    # Step 3: RNG transformation
    print("\n[3/3] Applying RNG transformation...")
    processed_windows = np.zeros_like(windows)
    for i in range(len(windows)):
        processed_windows[i] = transform_to_rng(windows[i], alpha=alpha, beta=beta)
    
    print(f"  ✓ RNG transformation complete")
    print(f"  ✓ Output shape: {processed_windows.shape}")
    print(f"  ✓ Value range: [{processed_windows.min():.3f}, {processed_windows.max():.3f}]")
    
    print(f"\n{'='*70}\n")
    
    return processed_windows, window_labels

print("✅ Preprocessing functions defined")

---
## Step 6: Preprocess All Datasets

In [ ]:
print("\n" + "🚀 "*35)
print("STARTING DATA PREPROCESSING PIPELINE")
print("🚀 "*35)

# Preprocessing parameters for Ninapro DB1
WINDOW_SIZE = 256  # 2.56 seconds at 100 Hz
STEP_SIZE = 50     # 0.5 second overlap
ALPHA = 2**13      # 8192 (for DB1)
BETA = 0.75
FS = 100          # Sampling frequency

# Preprocess training set
X_train, y_train = preprocess_dataset(
    train_emg, train_labels, 
    set_name="TRAINING SET",
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    alpha=ALPHA,
    beta=BETA,
    fs=FS
)

# Preprocess validation set
X_val, y_val = preprocess_dataset(
    val_emg, val_labels,
    set_name="VALIDATION SET",
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    alpha=ALPHA,
    beta=BETA,
    fs=FS
)

# Preprocess test set
X_test, y_test = preprocess_dataset(
    test_emg, test_labels,
    set_name="TEST SET",
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    alpha=ALPHA,
    beta=BETA,
    fs=FS
)

print("\n" + "✅ "*35)
print("PREPROCESSING COMPLETE!")
print("✅ "*35)

print(f"\n📊 FINAL DATASET SHAPES:")
print(f"  Training:   X={X_train.shape}, y={y_train.shape}")
print(f"  Validation: X={X_val.shape}, y={y_val.shape}")
print(f"  Test:       X={X_test.shape}, y={y_test.shape}")

---
## Step 7: Save Preprocessed Data to Disk

In [ ]:
print("\n" + "💾"*35)
print("SAVING PREPROCESSED DATA TO DISK")
print("💾"*35)

print(f"\nData directory: {os.path.abspath(PREPROCESSED_DATA_DIR)}")

# Save training data
print("\n[1/3] Saving training data...")
np.save(os.path.join(PREPROCESSED_DATA_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(PREPROCESSED_DATA_DIR, 'y_train.npy'), y_train)
print(f"  ✓ X_train: {X_train.shape}")
print(f"  ✓ y_train: {y_train.shape}")

# Save validation data
print("\n[2/3] Saving validation data...")
np.save(os.path.join(PREPROCESSED_DATA_DIR, 'X_val.npy'), X_val)
np.save(os.path.join(PREPROCESSED_DATA_DIR, 'y_val.npy'), y_val)
print(f"  ✓ X_val: {X_val.shape}")
print(f"  ✓ y_val: {y_val.shape}")

# Save test data
print("\n[3/3] Saving test data...")
np.save(os.path.join(PREPROCESSED_DATA_DIR, 'X_test.npy'), X_test)
np.save(os.path.join(PREPROCESSED_DATA_DIR, 'y_test.npy'), y_test)
print(f"  ✓ X_test: {X_test.shape}")
print(f"  ✓ y_test: {y_test.shape}")

print("\n" + "✅"*35)
print("DATA SAVED SUCCESSFULLY!")
print("✅"*35)

# Print file sizes
print("\nFile sizes:")
for filename in sorted(os.listdir(PREPROCESSED_DATA_DIR)):
    filepath = os.path.join(PREPROCESSED_DATA_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  {filename}: {size_mb:.2f} MB")

total_size = sum(os.path.getsize(os.path.join(PREPROCESSED_DATA_DIR, f)) 
                 for f in os.listdir(PREPROCESSED_DATA_DIR)) / (1024*1024)
print(f"\nTotal disk usage: {total_size:.2f} MB")

---
## Step 8: Clear Memory

In [ ]:
print("\n" + "🧹"*35)
print("CLEARING MEMORY")
print("🧹"*35)

# Delete large variables
del X_train, y_train, X_val, y_val, X_test, y_test
del train_emg, train_labels, train_subjects
del val_emg, val_labels, val_subjects
del test_emg, test_labels, test_subjects

# Force garbage collection
gc.collect()

print("✅ Memory cleared!")

---
## Step 9: Create Data Generator with On-the-Fly Augmentation

In [ ]:
class EMGDataGenerator(tf.keras.utils.Sequence):
    """
    Custom data generator that loads from disk and applies augmentation on-the-fly
    """
    def __init__(self, data_dir, dataset_name, batch_size=32, shuffle=True, augment=False):
        self.X_path = os.path.join(data_dir, f'X_{dataset_name}.npy')
        self.y_path = os.path.join(data_dir, f'y_{dataset_name}.npy')
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        
        # Load labels
        self.y = np.load(self.y_path)
        self.n_samples = len(self.y)
        self.indexes = np.arange(self.n_samples)
        
        # Load data in memory-mapped mode
        self.X = np.load(self.X_path, mmap_mode='r')
        
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __len__(self):
        return int(np.ceil(self.n_samples / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indexes = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = np.array(self.X[batch_indexes])
        y_batch = self.y[batch_indexes]
        
        if self.augment:
            X_batch = self._augment_batch(X_batch)
        
        return X_batch, {'main_output': y_batch, 'aux_output': y_batch}
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def _augment_batch(self, X_batch):
        augmented = []
        for window in X_batch:
            # Amplitude scaling
            if np.random.random() > 0.5:
                window = window * np.random.uniform(0.9, 1.1)
            
            # Gaussian noise
            if np.random.random() > 0.5:
                noise = np.random.normal(0, np.std(window) * 0.05, window.shape)
                window = window + noise
            
            # Channel dropout
            if np.random.random() > 0.7:
                channel_to_drop = np.random.randint(0, window.shape[1])
                window[:, channel_to_drop] = 0
            
            window = np.clip(window, 0, 1)
            augmented.append(window)
        
        return np.array(augmented)

print("✅ Data generator class defined")

---
## Step 10: Create Data Generators

In [ ]:
print("\n" + "⚙️"*35)
print("CREATING DATA GENERATORS")
print("⚙️"*35)

BATCH_SIZE = 64

# Training generator (with augmentation)
train_generator = EMGDataGenerator(
    data_dir=PREPROCESSED_DATA_DIR,
    dataset_name='train',
    batch_size=BATCH_SIZE,
    shuffle=True,
    augment=True
)

# Validation generator (no augmentation)
val_generator = EMGDataGenerator(
    data_dir=PREPROCESSED_DATA_DIR,
    dataset_name='val',
    batch_size=BATCH_SIZE,
    shuffle=False,
    augment=False
)

# Test generator (no augmentation)
test_generator = EMGDataGenerator(
    data_dir=PREPROCESSED_DATA_DIR,
    dataset_name='test',
    batch_size=BATCH_SIZE,
    shuffle=False,
    augment=False
)

print(f"\n✅ Generators created!")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training batches: {len(train_generator)}")
print(f"  Validation batches: {len(val_generator)}")
print(f"  Test batches: {len(test_generator)}")

# Test generator
X_sample, y_sample = train_generator[0]
print(f"\n  Sample batch shapes:")
print(f"    X: {X_sample.shape}")
print(f"    y (main): {y_sample['main_output'].shape}")
print(f"    y (aux): {y_sample['aux_output'].shape}")

---
## Step 11: Build HMSAR-Net Lite Model

In [ ]:
class ResidualBlock(layers.Layer):
    """
    Residual Block with skip connection
    """
    def __init__(self, filters, kernel_size, name_prefix='res'):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = layers.Conv1D(filters, kernel_size, padding='same', 
                                   name=f'{name_prefix}_conv1')
        self.bn1 = layers.BatchNormalization(name=f'{name_prefix}_bn1')
        self.relu1 = layers.ReLU(name=f'{name_prefix}_relu1')
        
        self.conv2 = layers.Conv1D(filters, kernel_size, padding='same', 
                                   name=f'{name_prefix}_conv2')
        self.bn2 = layers.BatchNormalization(name=f'{name_prefix}_bn2')
        
        self.shortcut_conv = None
        self.add = layers.Add(name=f'{name_prefix}_add')
        self.relu2 = layers.ReLU(name=f'{name_prefix}_relu2')
        
        self.filters = filters
    
    def build(self, input_shape):
        if input_shape[-1] != self.filters:
            self.shortcut_conv = layers.Conv1D(self.filters, 1, padding='same')
    
    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.bn1(x)
        x = self.relu1(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        
        shortcut = inputs
        if self.shortcut_conv is not None:
            shortcut = self.shortcut_conv(shortcut)
        
        x = self.add([x, shortcut])
        x = self.relu2(x)
        
        return x


def build_hmsar_net_lite(input_shape=(256, 10), num_classes=17):
    """
    Build HMSAR-Net Lite - Memory-efficient version
    """
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Multi-scale temporal features
    temp_fine = layers.Conv1D(8, 9, padding='same')(inputs)
    temp_fine = layers.BatchNormalization()(temp_fine)
    temp_fine = layers.ReLU()(temp_fine)
    temp_fine = layers.MaxPooling1D(4)(temp_fine)
    
    temp_med = layers.Conv1D(8, 17, padding='same')(inputs)
    temp_med = layers.BatchNormalization()(temp_med)
    temp_med = layers.ReLU()(temp_med)
    temp_med = layers.MaxPooling1D(4)(temp_med)
    
    temp_coarse = layers.Conv1D(8, 27, padding='same')(inputs)
    temp_coarse = layers.BatchNormalization()(temp_coarse)
    temp_coarse = layers.ReLU()(temp_coarse)
    temp_coarse = layers.MaxPooling1D(4)(temp_coarse)
    
    temporal_branch = layers.Concatenate()([temp_fine, temp_med, temp_coarse])
    
    # Channel synergy features
    chan_single = layers.Conv1D(8, 9, padding='same')(inputs)
    chan_single = layers.BatchNormalization()(chan_single)
    chan_single = layers.ReLU()(chan_single)
    chan_single = layers.MaxPooling1D(4)(chan_single)
    
    chan_pair = layers.Conv1D(8, 9, padding='same')(inputs)
    chan_pair = layers.BatchNormalization()(chan_pair)
    chan_pair = layers.ReLU()(chan_pair)
    chan_pair = layers.MaxPooling1D(4)(chan_pair)
    
    channel_branch = layers.Concatenate()([chan_single, chan_pair])
    
    # Attention-based fusion
    merged = layers.Concatenate()([temporal_branch, channel_branch])
    spatial_attention = layers.GlobalAveragePooling1D(keepdims=True)(merged)
    spatial_attention = layers.Dense(merged.shape[-1], activation='sigmoid')(spatial_attention)
    merged = layers.Multiply()([merged, spatial_attention])
    
    # Hierarchical residual blocks
    x = ResidualBlock(32, 9, name_prefix='res1_1')(merged)
    x = layers.Dropout(0.3)(x)
    x = layers.MaxPooling1D(4)(x)
    
    x = ResidualBlock(64, 9, name_prefix='res2_1')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.MaxPooling1D(2)(x)
    
    # Auxiliary classifier
    aux_gap = layers.GlobalAveragePooling1D(name='aux_gap')(x)
    aux_output = layers.Dense(num_classes, activation='softmax', name='aux_output')(aux_gap)
    
    # Adaptive global pooling
    gap = layers.GlobalAveragePooling1D()(x)
    gmp = layers.GlobalMaxPooling1D()(x)
    gsp = layers.Lambda(lambda x: K.std(x, axis=1))(x)
    
    global_features = layers.Concatenate()([gap, gmp, gsp])
    
    # Classification head
    dense1 = layers.Dense(128, activation='relu')(global_features)
    dense1 = layers.Dropout(0.5)(dense1)
    
    main_output = layers.Dense(num_classes, activation='softmax', name='main_output')(dense1)
    
    model = models.Model(
        inputs=inputs, 
        outputs=[main_output, aux_output],
        name='HMSAR_Net_Lite'
    )
    
    return model

print("✅ Model architecture functions defined")

---
## Step 12: Build and Compile Model

In [ ]:
print("\n" + "🏗️"*35)
print("BUILDING MODEL")
print("🏗️"*35)

INPUT_SHAPE = (256, 10)
NUM_CLASSES = 17

model = build_hmsar_net_lite(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)

print(f"\n✅ Model built!")
print(f"  Input shape: {INPUT_SHAPE}")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Total parameters: {model.count_params():,}")

# Display model summary
model.summary()

In [ ]:
print("\n" + "⚙️"*35)
print("COMPILING MODEL")
print("⚙️"*35)

LEARNING_RATE = 0.0001

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss={
        'main_output': 'sparse_categorical_crossentropy',
        'aux_output': 'sparse_categorical_crossentropy'
    },
    loss_weights={
        'main_output': 1.0,
        'aux_output': 0.3
    },
    metrics={
        'main_output': ['accuracy'],
        'aux_output': ['accuracy']
    }
)

print(f"\n✅ Model compiled!")
print(f"  Optimizer: Adam (lr={LEARNING_RATE})")
print(f"  Main loss weight: 1.0")
print(f"  Aux loss weight: 0.3")

---
## Step 13: Setup Callbacks

In [ ]:
print("\n" + "📋"*35)
print("SETTING UP CALLBACKS")
print("📋"*35)

early_stopping = EarlyStopping(
    monitor='val_main_output_accuracy',
    patience=50,
    restore_best_weights=True,
    verbose=1,
    mode='max'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_main_output_loss',
    factor=0.5,
    patience=20,
    min_lr=1e-7,
    verbose=1,
    mode='min'
)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, 'best_model.keras'),
    monitor='val_main_output_accuracy',
    save_best_only=True,
    save_weights_only=False,
    mode='max',
    verbose=1
)

callbacks = [early_stopping, reduce_lr, checkpoint]

print(f"\n✅ Callbacks configured:")
print(f"  - EarlyStopping (patience=50)")
print(f"  - ReduceLROnPlateau (factor=0.5, patience=20)")
print(f"  - ModelCheckpoint (save_best_only=True)")
print(f"  - Checkpoint path: {os.path.abspath(CHECKPOINT_DIR)}")

---
## Step 14: Train Model

In [ ]:
print("\n" + "🚀"*35)
print("STARTING TRAINING")
print("🚀"*35)

EPOCHS = 200

print(f"\nTraining configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Steps per epoch: {len(train_generator)}")
print(f"  Validation steps: {len(val_generator)}")
print("\n" + "="*70)
print("⏰ Training will start in 3 seconds...")
time.sleep(3)
print("🏃 GO!\n")

# Train with generators
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "✅"*35)
print("TRAINING COMPLETE!")
print("✅"*35)

---
## Step 15: Visualize Training History

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Main output accuracy
    axes[0, 0].plot(history.history['main_output_accuracy'], label='Train', linewidth=2)
    axes[0, 0].plot(history.history['val_main_output_accuracy'], label='Val', linewidth=2)
    axes[0, 0].set_title('Main Output Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Main output loss
    axes[0, 1].plot(history.history['main_output_loss'], label='Train', linewidth=2)
    axes[0, 1].plot(history.history['val_main_output_loss'], label='Val', linewidth=2)
    axes[0, 1].set_title('Main Output Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Auxiliary output accuracy
    axes[1, 0].plot(history.history['aux_output_accuracy'], label='Train', linewidth=2)
    axes[1, 0].plot(history.history['val_aux_output_accuracy'], label='Val', linewidth=2)
    axes[1, 0].set_title('Auxiliary Output Accuracy', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Total loss
    axes[1, 1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[1, 1].plot(history.history['val_loss'], label='Val', linewidth=2)
    axes[1, 1].set_title('Total Loss', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(RESULTS_DIR, 'training_history.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved training history to: {save_path}")
    plt.show()

# Plot history
plot_training_history(history)

# Print best metrics
print("\n" + "📊"*35)
print("TRAINING METRICS")
print("📊"*35)

best_val_acc = max(history.history['val_main_output_accuracy'])
best_epoch = history.history['val_main_output_accuracy'].index(best_val_acc) + 1

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"Best Epoch: {best_epoch}")
print(f"Final Training Accuracy: {history.history['main_output_accuracy'][-1]*100:.2f}%")
print(f"Final Validation Accuracy: {history.history['val_main_output_accuracy'][-1]*100:.2f}%")
print(f"Total Epochs Trained: {len(history.history['loss'])}")

---
## Step 16: Evaluate on Test Set

In [ ]:
print("\n" + "🧪"*35)
print("EVALUATING ON TEST SET")
print("🧪"*35)

# Load best model
best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.keras')
print(f"\nLoading best model from: {best_model_path}")
model = tf.keras.models.load_model(best_model_path, custom_objects={'ResidualBlock': ResidualBlock})

# Evaluate
test_results = model.evaluate(test_generator, verbose=1)

print("\n" + "="*70)
print("TEST SET RESULTS")
print("="*70)

# Extract metrics
metric_names = model.metrics_names
for name, value in zip(metric_names, test_results):
    if 'accuracy' in name:
        print(f"{name}: {value*100:.2f}%")
    else:
        print(f"{name}: {value:.4f}")

---
## Step 17: Generate Predictions and Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("\n" + "📈"*35)
print("GENERATING PREDICTIONS")
print("📈"*35)

# Get predictions
predictions = model.predict(test_generator, verbose=1)
y_pred_main = np.argmax(predictions[0], axis=1)

# Get true labels
y_true = np.load(os.path.join(PREPROCESSED_DATA_DIR, 'y_test.npy'))

# Calculate confusion matrix
cm = confusion_matrix(y_true, y_pred_main)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(17), yticklabels=range(17))
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
cm_path = os.path.join(RESULTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
print(f"Saved confusion matrix to: {cm_path}")
plt.show()

# Print classification report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_true, y_pred_main, 
                          target_names=[f'Gesture {i}' for i in range(17)]))

---
## Step 18: Per-Class Accuracy Analysis

In [ ]:
# Calculate per-class accuracy
per_class_acc = []
for i in range(17):
    mask = y_true == i
    if np.sum(mask) > 0:
        acc = np.mean(y_pred_main[mask] == y_true[mask])
        per_class_acc.append(acc)
    else:
        per_class_acc.append(0)

# Plot per-class accuracy
plt.figure(figsize=(12, 6))
bars = plt.bar(range(17), [acc*100 for acc in per_class_acc], color='steelblue', alpha=0.8)
plt.axhline(y=np.mean(per_class_acc)*100, color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {np.mean(per_class_acc)*100:.2f}%')
plt.xlabel('Gesture Class', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Per-Class Accuracy - Test Set', fontsize=16, fontweight='bold')
plt.xticks(range(17))
plt.ylim([0, 105])
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
acc_path = os.path.join(RESULTS_DIR, 'per_class_accuracy.png')
plt.savefig(acc_path, dpi=150, bbox_inches='tight')
print(f"Saved per-class accuracy to: {acc_path}")
plt.show()

# Print statistics
print("\n" + "="*70)
print("PER-CLASS ACCURACY STATISTICS")
print("="*70)
for i, acc in enumerate(per_class_acc):
    print(f"Gesture {i:2d}: {acc*100:6.2f}%")
print("\n" + "-"*70)
print(f"Mean Accuracy: {np.mean(per_class_acc)*100:.2f}%")
print(f"Std Deviation: {np.std(per_class_acc)*100:.2f}%")
print(f"Min Accuracy:  {np.min(per_class_acc)*100:.2f}%")
print(f"Max Accuracy:  {np.max(per_class_acc)*100:.2f}%")

---
## Step 19: Save Final Results Summary

In [ ]:
# Create results summary
results_summary = {
    'Model': 'HMSAR-Net Lite',
    'Dataset': 'Ninapro DB1 Exercise 2',
    'Evaluation': 'Subject-Independent',
    'Train Subjects': 's1-s20',
    'Val Subjects': 's21-s24',
    'Test Subjects': 's25-s27',
    'Total Parameters': model.count_params(),
    'Best Val Accuracy': f"{best_val_acc*100:.2f}%",
    'Test Accuracy': f"{np.mean(y_pred_main == y_true)*100:.2f}%",
    'Mean Per-Class Accuracy': f"{np.mean(per_class_acc)*100:.2f}%",
    'Epochs Trained': len(history.history['loss']),
    'Best Epoch': best_epoch
}

# Save to file
summary_path = os.path.join(RESULTS_DIR, 'results_summary.txt')
with open(summary_path, 'w') as f:
    f.write("="*70 + "\n")
    f.write("HMSAR-NET RESULTS SUMMARY\n")
    f.write("="*70 + "\n\n")
    for key, value in results_summary.items():
        f.write(f"{key}: {value}\n")
    f.write("\n" + "="*70 + "\n")

print("\n" + "💾"*35)
print("RESULTS SAVED")
print("💾"*35)
print(f"\nResults directory: {os.path.abspath(RESULTS_DIR)}")
print("\nSaved files:")
for filename in os.listdir(RESULTS_DIR):
    print(f"  - {filename}")

print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
for key, value in results_summary.items():
    print(f"{key}: {value}")
print("="*70)

---
## 🎉 Pipeline Complete!

**What we accomplished:**
1. ✅ Loaded Ninapro DB1 Exercise 2 dataset (27 subjects, 17 gestures)
2. ✅ Applied subject-independent split (Train: s1-s20, Val: s21-s24, Test: s25-s27)
3. ✅ Preprocessed with bandpass/notch filtering and RNG transformation
4. ✅ Created sliding windows (256 samples, 50 sample step)
5. ✅ Saved preprocessed data to disk for memory efficiency
6. ✅ Built HMSAR-Net Lite with multi-scale features, attention, and deep supervision
7. ✅ Trained with on-the-fly augmentation and composite loss
8. ✅ Evaluated on held-out test subjects
9. ✅ Generated comprehensive results and visualizations

**Next Steps:**
- Compare with baseline methods (SVM, LDA, simple CNN)
- Test on other Ninapro databases (DB2, DB3, DB4)
- Implement transfer learning experiments
- Try few-shot adaptation for new subjects

**Files Generated:**
- `./preprocessed_data/` - Preprocessed datasets (.npy files)
- `./model_checkpoints/` - Best trained model (.keras file)
- `./results/` - Training curves, confusion matrix, accuracy plots, summary